# Modeling

In [200]:
import pandas as pd
from sklearn.decomposition import PCA
import numpy as np

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)

#import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn import model_selection
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV


from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.metrics import root_mean_squared_log_error
from numpy import expm1
from pathlib import Path

In [201]:
# PATH DEFINITIONS
BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data"
MODEL_DATA_DIR = DATA_DIR / "data_model"

In [202]:
df_train = pd.read_csv(MODEL_DATA_DIR / "train_model.csv")

In [203]:
X_train = df_train.drop(columns=['SALEPRICE'])
y_train = df_train['SALEPRICE']

In [204]:
df_train.head()

,NEIGHBORHOOD,OVERALLQUAL,EXTERQUAL,BSMTQUAL,HEATINGQC,GRLIVAREA,KITCHENQUAL,TOTRMSABVGRD,GARAGEYRBLT,GARAGEFINISH,...,EXTERIOR1ST_OTHER,EXTERIOR1ST_PLYWOOD,EXTERIOR1ST_VINYLSD,EXTERIOR1ST_WD SDNG,GARAGETYPE_DETCHD,GARAGETYPE_OTHER,SALETYPE_CON,SALETYPE_NEW,SALETYPE_WD,SALEPRICE
0,0.487709,0.545790,0.932643,0.496551,0.813156,0.517908,0.634784,0.94979,0.914436,0.127375,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.247699
1,1.144719,-0.234112,-0.821614,0.496551,0.813156,-0.475000,-0.935844,-0.34389,-0.279934,0.127375,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,12.109016
2,0.487709,0.545790,0.932643,0.496551,0.813156,0.660048,0.634784,-0.34389,0.825964,0.127375,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.317171
3,1.801730,1.325692,0.932643,0.496551,0.813156,1.338577,0.634784,1.59663,0.781728,0.127375,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.429220
4,0.980466,1.325692,0.932643,2.001168,0.813156,0.487180,0.634784,0.30295,0.958671,0.127375,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,12.634606


In [205]:
pca_analysis = PCA(random_state=0) 
pca_analysis.fit(X_train)

individual_variance = pca_analysis.explained_variance_ratio_
accumulated_variance = individual_variance.cumsum()

pca_table = pd.DataFrame({
    'COMPONENT': range(1, len(accumulated_variance) + 1),
    'INDIVIDUAL_VARIANCE': (individual_variance).round(4),
    'ACCUMULATED_VARIANCE': (accumulated_variance).round(4),
    'DELTA': (pd.Series(accumulated_variance).diff().fillna(accumulated_variance[0]) * 100).round(4)
})

first_above_95 = np.argmax(np.array(pca_table['ACCUMULATED_VARIANCE']) >= 0.95)
first_above_95

np.int64(16)

In [206]:
pca = PCA(n_components=16, random_state=0)
X_train_pca = pca.fit_transform(X_train) 

In [207]:
models = {
    'Ridge':             Ridge(random_state=0),
    'Lasso':             Lasso(random_state=0),
    'ElasticNet':        ElasticNet(random_state=0),
    'Bayesian Ridge':    BayesianRidge(),
    'Huber':             HuberRegressor(max_iter=500),
    'KNN Regressor':     KNeighborsRegressor(),
    'SVR':               SVR(),
    'Random Forest':     RandomForestRegressor(random_state=0),
    'Extra Trees':       ExtraTreesRegressor(random_state=0),
    'Gradient Boosting': GradientBoostingRegressor(random_state=0),
}

kfold = model_selection.KFold(n_splits=5, shuffle=True, random_state=0)

In [208]:
results = []
for name, model in models.items():
    pipeline = Pipeline([
        ('model', model)
    ])
    scores = model_selection.cross_validate(
        pipeline,
        X_train,
        y_train,
        scoring='neg_root_mean_squared_log_error',
        cv=kfold,
        n_jobs=-1
    )
    results.append({
        'MODEL': name.upper(),
        'RMSLE_MEAN': -scores['test_score'].mean(),
        'RMSLE_STD':   scores['test_score'].std()
    })

In [209]:
df_exploration = pd.DataFrame(results)
df_exploration.columns = df_exploration.columns.str.upper()
df_exploration = df_exploration.round(4)

df_exploration = df_exploration.sort_values(by='RMSLE_MEAN').reset_index(drop=True)

df_exploration

,MODEL,RMSLE_MEAN,RMSLE_STD
0,GRADIENT BOOSTING,0.0091,0.0006
1,EXTRA TREES,0.0093,0.0008
2,RANDOM FOREST,0.0094,0.0005
3,RIDGE,0.0097,0.0007
4,BAYESIAN RIDGE,0.0097,0.0007
5,HUBER,0.0098,0.0009
6,SVR,0.0098,0.0008
7,KNN REGRESSOR,0.0109,0.0005
8,LASSO,0.0265,0.0014
9,ELASTICNET,0.0265,0.0014


## GRADIENT BOOSTING

In [210]:
df_exploration_analysis = df_exploration.copy()
df_exploration_analysis[['RMSLE_MEAN', 'RMSLE_STD']] = df_exploration_analysis[['RMSLE_MEAN', 'RMSLE_STD']].rank(0, numeric_only=True, method='min', ascending=True)
df_exploration_analysis['POINTS'] = df_exploration_analysis['RMSLE_MEAN'] + df_exploration_analysis['RMSLE_STD'] 

df_exploration_analysis = df_exploration_analysis.sort_values(by='POINTS', ascending=True).reset_index(drop=True)
df_exploration_analysis

,MODEL,RMSLE_MEAN,RMSLE_STD,POINTS
0,GRADIENT BOOSTING,1.0,3.0,4.0
1,RANDOM FOREST,3.0,1.0,4.0
2,EXTRA TREES,2.0,6.0,8.0
3,RIDGE,4.0,4.0,8.0
4,BAYESIAN RIDGE,4.0,4.0,8.0
5,KNN REGRESSOR,8.0,1.0,9.0
6,SVR,6.0,6.0,12.0
7,HUBER,6.0,8.0,14.0
8,LASSO,9.0,9.0,18.0
9,ELASTICNET,9.0,9.0,18.0


In [211]:
gb_model = GradientBoostingRegressor(random_state=0)

param_distributions = {
    "n_estimators": [500, 525, 550],
    "learning_rate": [0.045, 0.05, 0.06],
    "max_depth": [6, 7], # experimentacao
    "min_samples_split": [90, 100, 110], #dobro do min_sample_leaf
    "min_samples_leaf": [45, 50, 55], # Geralmente, na faixa de 1-5%
    "max_features": [0.8, 0.85, 0.90], # um valor classico é \sqrt (no caso, 4 = 0.25)
    "subsample": [0.80, 0.85, 0.90] # depende do tamanho do modelo, cria generalização
}

random_search = RandomizedSearchCV(
    estimator=gb_model,
    param_distributions=param_distributions,
    n_jobs=-1,
    random_state=0,
    scoring='neg_root_mean_squared_log_error'
)

random_search.fit(X_train_pca, y_train)

model_best_gb = random_search.best_estimator_
model_best_gb_score = random_search.best_score_

print(model_best_gb_score)
print(model_best_gb)

-0.009766701853284173
GradientBoostingRegressor(learning_rate=0.05, max_depth=6, max_features=0.85,
                          min_samples_leaf=50, min_samples_split=90,
                          n_estimators=525, random_state=0, subsample=0.9)


In [ ]:
param_grid_refined = {
    'learning_rate': [0.05],
    'max_depth':     [4, 5, 6],
    'max_features':  [1.0],
    'min_samples_leaf':  [2, 4, 6],
    'min_samples_split': [100, 120, 140],
    'n_estimators':  [225, 275, 325],
    'subsample':     [0.45, 0.55],
}

grid_search = GridSearchCV(
    estimator=gb_model,
    param_grid=param_grid_refined,
    n_jobs=-1,
    scoring='neg_root_mean_squared_log_error'
)

grid_search.fit(X_train_pca, y_train)

model_best_gb = grid_search.best_estimator_
model_best_gb_score = grid_search.best_score_

print(model_best_gb_score)
print(model_best_gb)

-0.009322610768679452
GradientBoostingRegressor(learning_rate=0.05, max_depth=5, max_features=1.0,
                          min_samples_leaf=5, min_samples_split=120,
                          n_estimators=275, random_state=0, subsample=0.55)


In [213]:
y_pred = model_best_gb.predict(X_train_pca)
print(round(root_mean_squared_log_error(y_true=y_train, y_pred=y_pred), 4))

0.0054


## RANDOM FOREST

In [ ]:
rf_model = RandomForestRegressor(random_state=0)

